In [ ]:
import pandas as pd

data = pd.read_csv("data/ChargePointData2011-2020-ManullyPreCleaned3.csv", low_memory=False)
data

In [ ]:
data.info()

In [ ]:
# import matplotlib.pyplot as plt
# data.hist(bins=50, figsize=(20,15))
# plt.show()

In [ ]:
# What does the latitude and longitude represent? It is not equal to each station
'''
Feature engineer turn Start Date into:
# Hour of Day: Captures peak commute times.
# Day of Week: Captures the difference between a Monday workday and a Sunday leisure day.
# Month/Season: Captures holiday travel or weather-related usage.
# Is Weekend: A binary flag (0 or 1) that simplifies the weekend pattern for the model (?)
# Is holiday: 
機器學習用上面的數據，也用：
# MAC Address
# Port Type / Plug Type: Some cars only use specific plugs; if a station has a rare plug, it might be less busy.
# Fee: As seen in your data correlation (~0.33), the price often influences how long people stay. Higher fees sometimes lead to shorter total durations.
# Charging Time (hh:mm:ss)
# Note: Port Type missing some values, but just 9
# Extra note: Other column not on remove data list will be used as well (?)

Data that could be important
# Fee
# Energy (kWh) (?): Data leak

Remove data that is not important to the model
# Port type and plug type is the same (each data equal to each other), so just pick one, I picked plug type (port type missing values)
# Station name, Org Name, Start Time Zone, End Date, End Time Zone, Transaction Date (Pacific Time), 
# EVSE ID, Address 1, City, State/Province, Postal Code, Country, Country
# Longitude, latitude
# Currency, Ended By, Plug In Event Id, Driver Postal Code, User ID, System S/N, Model Number
# Port Number (?),


Label: Total Duration

Two ways to approach:
Classification First: Predict a binary label: Available (0) or Occupied (1).
# How to get data that gives the label 0? Just get random time in the world?
# Calculate how many rows you have for Label 1. Generate the exact same number of rows for Label 0 by sampling repeatedly from your gaps by 
Negative Sampling (If a station is empty from 2:00 PM to 4:00 PM, you can create a training row for "3:00 PM" with a label of 0).
# Use Strategic Distribution Sampling ensures your training data has plenty of examples of "Available at 3:00 AM," "Available at 4:00 AM," etc. 
Flow
# Model A (Classifier): Asks "Is there a line?".
# Conditional Logic: If No (0), Wait Time = 0. If Yes (1), call Model B.
# Model B (Regressor): Asks "Since there is a line, how long until the current session ends?".

Regression Second: If the station is predicted to be occupied, run a second model to predict the Remaining Total Duration based on the features above
This is generally the standard for IT and Engineering projects because it mirrors real-world decision-making

Save the MAC and keys (Port Type / Plug Type, Fee, Average Energy per Session) even future events on another database, so it can be called
when putting inputs called
'''

In [ ]:
import numpy as np
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.preprocessing import TargetEncoder 

def to_seconds(time_str):
    if pd.isna(time_str): return 0
    h, m, s = map(int, str(time_str).split(':'))
    return (h * 60 * 60) + (m * 60) + s


def engineer_time_features(df):
    # Ensure column is in datetime format
    df['Start Date'] = pd.to_datetime(df['Start Date'])
    
    # 1. Simple Extraction
    df['Minute'] = df['Start Date'].dt.minute
    df['Hour'] = df['Start Date'].dt.hour
    df['DayOfWeek'] = df['Start Date'].dt.dayofweek # 0=Mon, 6=Sun
    df['Month'] = df['Start Date'].dt.month
    
    # 2. Binary Flags
    df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
    
    # 3. Holiday Identification
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df['Start Date'].min(), end=df['Start Date'].max())
    df['IsHoliday'] = df['Start Date'].dt.normalize().isin(holidays).astype(int)
    
    # 4. Cyclical Encoding (Trigonometry)
    # This prevents the "23:00 to 00:00" numerical jump
    # Used for Neural network, can be removed for normal ML
    df['Hour_Sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
    df['Hour_Cos'] = np.cos(2 * np.pi * df['Hour'] / 24)

    df = df.drop(columns=['Start Date'])
    
    return df
    


def clean_dataset(data):
    data = data.copy()
    cleaned_data = data[[
        'MAC Address', 'Start Date'
    ]]
    cleaned_data = engineer_time_features(cleaned_data.copy())
    
    cleaned_data['Plug Type Encoded'] = data['Plug Type'].map({'J1772': 0, 'NEMA 5-20R': 1})
    cleaned_data['Total_Duration_Secs'] = data['Total Duration (hh:mm:ss)'].apply(to_seconds)
    cleaned_data['Availability'] = 1

    # cleaned_data['Price_per_Min'] = cleaned_data.apply(lambda x: x['Fee'] / x['Duration_Secs'] if x['Duration_Secs'] > 0 else 0, axis=1)
    
    return cleaned_data

In [ ]:
cleaned_data = clean_dataset(data)
cleaned_data

In [ ]:
cleaned_data['Plug Type Encoded'].value_counts()

In [ ]:
# Do this on a dataset that is already split, OPEN PAST NOTES

# encoder = TargetEncoder(smooth="auto")
# X = data[['MAC Address']]
# y = cleaned_data['Total_Duration_Secs']
# cleaned_data['MAC_Encoded'] = encoder.fit_transform(X, y)

In [ ]:
import pandas as pd
import numpy as np
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.preprocessing import TargetEncoder

def to_seconds_vectorized(series):
    """Converts hh:mm:ss to seconds using fast vectorized pandas math."""
    temp = series.str.split(':', expand=True).astype(int)
    return (temp[0] * 3600) + (temp[1] * 60) + temp[2]

def engineer_time_features(df):
    """Extracts temporal signals and handles cyclical encoding."""
    # Ensure Start Date is datetime for extraction
    df['Start Date'] = pd.to_datetime(df['Start Date'])
    
    df['Minute'] = df['Start Date'].dt.minute
    df['Hour'] = df['Start Date'].dt.hour
    df['DayOfWeek'] = df['Start Date'].dt.dayofweek
    df['Month'] = df['Start Date'].dt.month
    df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
    
    # Fast Holiday lookup
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df['Start Date'].min(), end=df['Start Date'].max())
    df['IsHoliday'] = df['Start Date'].dt.normalize().isin(holidays).astype(int)
    
    # Cyclical Encoding for Neural Networks
    df['Hour_Sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
    df['Hour_Cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
    
    return df

def build_final_dataset(data):
    """Main pipeline: Generates available rows and sorts chronologically."""
    data = data.copy()
    data['Start Date'] = pd.to_datetime(data['Start Date'])
    data['End Date'] = pd.to_datetime(data['End Date'])
    
    # 1. Sort initially to detect gaps per pole
    data = data.sort_values(['MAC Address', 'Start Date'])
    
    # 2. Extract Occupied Sessions (Label 1)
    occupied = data[['MAC Address', 'Start Date', 'End Date', 'Plug Type']].copy()
    occupied['Total_Duration_Secs'] = to_seconds_vectorized(data['Total Duration (hh:mm:ss)'])
    occupied['Availability'] = 1
    
    # 3. Generate Available Sessions (Label 0) between gaps
    available_rows = []
    for mac, group in data.groupby('MAC Address'):
        group = group.sort_values('Start Date')
        for i in range(len(group) - 1):
            curr_end = group.iloc[i]['End Date']
            next_start = group.iloc[i+1]['Start Date']
            
            # Identify idle gaps longer than 5 minutes
            if next_start > curr_end + pd.Timedelta(minutes=5):
                available_rows.append({
                    'MAC Address': mac,
                    'Start Date': curr_end,
                    'End Date': next_start,
                    'Plug Type': group.iloc[i]['Plug Type'],
                    'Total_Duration_Secs': (next_start - curr_end).total_seconds(),
                    'Availability': 0
                })
    
    available_df = pd.DataFrame(available_rows)
    
    # 4. Combine and Sort Chronologically for clean splitting
    full_df = pd.concat([occupied, available_df], ignore_index=True)
    full_df = full_df.sort_values('Start Date').reset_index(drop=True)
    
    # 5. Apply Feature Engineering
    full_df = engineer_time_features(full_df)
    
    # 6. Final Encoding
    full_df['Plug Type Encoded'] = full_df['Plug Type'].map({'J1772': 0, 'NEMA 5-20R': 1}).fillna(0).astype(int)
    
    # We keep 'Start Date' for the split, but drop strings
    return full_df.drop(columns=['End Date', 'Plug Type'])

In [ ]:
final_df = build_final_dataset(data)

In [ ]:
final_df

In [ ]:
final_df['Availability'].value_counts()

In [ ]:
split_idx = int(len(final_df) * 0.8)
train_df = final_df.iloc[:split_idx]
test_df = final_df.iloc[split_idx:]
train_df

In [ ]:
train_df['Availability'].value_counts()

In [ ]:
test_df['Availability'].value_counts()

In [ ]:
test_df

## Separating data and label (Availability)

In [ ]:
train_df_label_availability = train_df['Availability'].copy()
train_df_label_duration = train_df['Total_Duration_Secs'].copy()
train_df_data = train_df.drop(['Availability','Total_Duration_Secs'], axis = 1)

In [ ]:
train_df_label_duration

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class CustomAttributeTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    def fit(self, X, y=None):
        return self
    def transform(self, data):
        data_copy = data.copy()
        data_copy = data_copy.drop(columns=['Start Date'])

        print(data_copy)
        return data_copy

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import TargetEncoder

cat_pipeline = Pipeline([
    ("targetencoder", TargetEncoder())
])

In [ ]:
from sklearn.compose import ColumnTransformer

num_attribs = list(train_df_data.select_dtypes(include="number").columns)
print(num_attribs)

cat_attribs = ["MAC Address"]

full_pipeline = Pipeline([
    ('custom', CustomAttributeTransformer()),
    ('preprocess', ColumnTransformer([
        ("cat", cat_pipeline, cat_attribs)
    ])),
    ('std_scaler', StandardScaler())
])
# print(train_df_data)
train_df_prepared = full_pipeline.fit_transform(train_df_data, train_df_label_availability)
train_df_prepared

In [ ]:
ct = full_pipeline.named_steps['preprocess']
training_dataset = pd.DataFrame(
    train_df_prepared, columns= ct.get_feature_names_out(), index=train_df_data.index
)
training_dataset

In [ ]:
training_dataset['num__Plug Type Encoded'].value_counts()

## Training a Model (Model A availability)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

log_reg = LogisticRegression()
log_reg.fit(training_dataset, train_df_label_availability)

log_reg_scores = cross_val_score(log_reg, training_dataset, train_df_label_availability, scoring="accuracy", cv=5)
print("Scores:", log_reg_scores)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dec_tree = DecisionTreeClassifier()
dec_tree.fit(training_dataset, train_df_label_availability)
dec_tree_scores = cross_val_score(dec_tree, training_dataset, train_df_label_availability, scoring="accuracy", cv=5)
print("Scores:", dec_tree_scores)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rd_forest = RandomForestClassifier()
rd_forest.fit(training_dataset, train_df_label_availability)
rd_forest_scores = cross_val_score(rd_forest, training_dataset, train_df_label_availability, scoring="accuracy", cv=5, n_jobs=-1)
print("Scores:", rd_forest_scores)